In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
cat = Table.read('/global/cfs/cdirs/desi/users/rongpu/spectro/sv1/sv1-exposures_20210224.fits')
print(len(cat))

tileid_list = [80605, 80606, 80607, 80608, 80609, 80610, 80613]

mask = np.in1d(cat['TILEID'], tileid_list)
cat = cat[mask]
print(len(cat), len(np.unique(cat['EXPID'])))
cat.sort('EXPID')

cat[:1]

1243
161 161


NIGHT,EXPID,FIELD,TARGETS,OBSCONDITIONS,ARIZONA_TIMEOBS,EBV,SPECMODEL_SKY_GMAG_AB,SPECMODEL_SKY_RMAG_AB,SPECMODEL_SKY_ZMAG_AB,GFA_ORIGIN,B_DEPTH,R_DEPTH,Z_DEPTH,B_DEPTH_EBVAIR,R_DEPTH_EBVAIR,Z_DEPTH_EBVAIR,DAILY_BITPSFFN,DAILY_BITFRAMEFN,DAILY_BITSKYFN,DAILY_BITSFRAMEFN,DAILY_BITFLUXCALIBFN,DAILY_BITCFRAMEFN,TGT,SKY,STD,WD,LRG,ELG,QSO,BGS,MWS,TILEID,TILERA,TILEDEC,EXPTIME,MJDOBS,SKYMON_NEXP,SKYMON_SKYCAM0_MEAN,SKYMON_SKYCAM0_MEAN_ERR,SKYMON_SKYCAM1_MEAN,SKYMON_SKYCAM1_MEAN_ERR,SKYMON_AVERAGE_MEAN,SKYMON_AVERAGE_MEAN_ERR,GFA_AIRMASS,GFA_MOON_ILLUMINATION,GFA_MOON_ZD_DEG,GFA_MOON_SEP_DEG,GFA_TRANSPARENCY,GFA_FWHM_ASEC,GFA_SKY_MAG_AB,GFA_FIBER_FRACFLUX,GFA_FIBER_FRACFLUX_ELG,GFA_TRANSPFRAC,GFA_MAXCONTRAST,GFA_MINCONTRAST,GFA_KTERM,GFA_RADPROF_FWHM_ASEC,GFA_FIBERFAC,GFA_FIBERFAC_ELG,EPHEM_NOON,EPHEM_DUSK,EPHEM_DAWN,EPHEM_BRIGHTDUSK,EPHEM_BRIGHTDAWN,EPHEM_BRIGHTDUSK_LST,EPHEM_BRIGHTDAWN_LST,EPHEM_MOONRISE,EPHEM_MOONSET,EPHEM_MOON_ILLUM_FRAC,EPHEM_NEAREST_FULL_MOON
int64,int64,bytes30,bytes16,int16,bytes19,float32,float32,float32,float32,bytes13,float32,float32,float32,float32,float32,float32,int64,int64,int64,int64,int64,int64,int16,int16,int16,int16,int16,int16,int16,int16,int16,int64,float32,float32,float32,float32,int16,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
20201214,67710,XMM-LSS,QSO+LRG,1,2020-12-14T21:12:40,0.026,22.46653,21.36987,19.686352,matched_coadd,19.1,17.0,18.6,14.928702,14.590197,17.078018,1073741823,1073741823,1073741823,1073741823,0,0,4200,800,136,13,2117,1341,1428,464,191,80605,36.448,-4.601,900.0,59198.176,0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,1.2455698,0.0036080289,133.7576,119.15718,0.8552955,3.5648944,20.274342,0.09248331,0.08644419,0.07910056,4.109576,3.6593802,0.114,3.7471519,0.1273114,0.1655635,59197.791666666664,59198.07704947222,59198.53638188851,59198.05633958238,59198.557098368896,-7.152731532294979,173.61402820025236,59197.59470830986,59198.02745986081,0.005432771706774475,14.852906347514363


In [4]:
# David Schlegel's list of exposures with speed == r_depth / exptime > 0.3
explist_dark = Table.read('/global/cfs/cdirs/desi/users/rongpu/spectro/sv1/explist-deep-dark.txt', format='ascii')
explist_bright = Table.read('/global/cfs/cdirs/desi/users/rongpu/spectro/sv1/explist-deep-bright.txt', format='ascii')

explist = vstack([explist_dark, explist_bright], join_type='exact')
print(len(explist))

explist.sort('EXPID')

explist[:1]

96


TILEID,NIGHT,EXPID
int64,int64,int64
80607,20201214,67744


In [5]:
speed = cat['R_DEPTH'] / cat['EXPTIME']
mask = speed > 0.3
print(np.sum(mask))

np.all(explist['EXPID']==cat['EXPID'][mask])

cat = cat[mask]
print(len(cat))

96
96


In [6]:
for tileid in np.unique(cat['TILEID']):
    mask = cat['TILEID']==tileid
    print(tileid, np.sum(cat['R_DEPTH_EBVAIR'][mask]), np.sum(mask), cat['TARGETS'][mask][0], cat['FIELD'][mask][0])

80605 6132.751 11 QSO+LRG XMM-LSS
80606 6628.393 11 ELG XMM-LSS
80607 9999.572 17 QSO+LRG Lynx
80608 13990.975 20 ELG Lynx
80609 7011.713 13 QSO+LRG COSMOS
80610 9014.408 15 ELG COSMOS
80613 2384.144 9 BGS+MWS Lynx


In [7]:
for tileid in np.unique(cat['TILEID']):
    mask = cat['TILEID']==tileid
    print(tileid, np.sum(cat['R_DEPTH_EBVAIR'][mask]), np.sum(cat['R_DEPTH_EBVAIR'][mask])/4000.)

80605 6132.751 1.533187744140625
80606 6628.393 1.6570982666015626
80607 9999.572 2.49989306640625
80608 13990.975 3.49774365234375
80609 7011.713 1.75292822265625
80610 9014.408 2.25360205078125
80613 2384.144 0.5960360107421875


In [8]:
print('TILEID n_exp tot_depth n_sub_min n_sub_max  n_sub')
for tileid in np.unique(cat['TILEID']):
    if tileid==80613:
        nom_depth = 600.
    else:
        nom_depth = 4000.
    mask = cat['TILEID']==tileid
    tot_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    n_sub_min = tot_depth/nom_depth
    if n_sub_min < 2:
        margin = 0.15
    else:
        margin = 0.1
    n_sub_max = tot_depth/(nom_depth*(1-margin))
    n_sub = int(np.floor(n_sub_max))
    print(tileid, '{:3} {:9.2f} {:6.2f} {:6.2f} {:3}'.format(np.sum(mask), tot_depth, n_sub_min, n_sub_max, n_sub))

TILEID n_exp tot_depth n_sub_min n_sub_max  n_sub
80605  11   6132.75   1.53   1.80   1
80606  11   6628.39   1.66   1.95   1
80607  17   9999.57   2.50   2.78   2
80608  20  13990.97   3.50   3.89   3
80609  13   7011.71   1.75   2.06   2
80610  15   9014.41   2.25   2.50   2
80613   9   2384.14   3.97   4.42   4


----

In [9]:
np.random.seed(511)

subsamp_dict = {}

# tileid = 80607
for tileid in np.unique(cat['TILEID']):
    
    print()
    
    if tileid==80613:
        nom_depth = 600.
    else:
        nom_depth = 4000.
    mask = cat['TILEID']==tileid

    n_exp = np.sum(mask)

    tot_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    n_sub_min = tot_depth/nom_depth
    if n_sub_min < 2:
        margin = 0.15
    else:
        margin = 0.1
    n_sub_max = tot_depth/(nom_depth*(1-margin))
    n_sub = int(np.floor(n_sub_max))
    print(tileid, '{:3} {:9.2f} {:6.2f} {:6.2f} {:3}'.format(np.sum(mask), tot_depth, n_sub_min, n_sub_max, n_sub))

    mask = cat['TILEID']==tileid
    expid_list = np.array(cat['EXPID'][mask])
    subsets = []

    total_iteration = 0
    while True:
        total_iteration += 1
        success = False
        iteration = 0
        while success==False:
            iteration += 1
            subset = []
            subset_depth = []
            while True:
                mask = ~np.in1d(expid_list, subset)
                expid = np.random.choice(expid_list[mask])
                subset.append(expid)
                subset_depth.append(cat['R_DEPTH_EBVAIR'][cat['EXPID']==expid][0])

                if total_iteration<200:
                    margin = 0.05
                elif ((total_iteration>200) & (total_iteration<400)):
                    margin = 0.1
                elif (total_iteration>400):
                    margin = 0.15
                if (np.sum(subset_depth)>(1-margin)*nom_depth) and (np.sum(subset_depth)<(1+margin)*nom_depth):
    #             if (np.sum(subset_depth)>0.8*nom_depth) and (np.sum(subset_depth)<1.2*nom_depth):
                    success = True
                    # print(np.sum(subset_depth))
                    break
                if len(subset)==len(expid_list):
                    break
            if iteration>(np.minimum(100, np.math.factorial(n_exp))):
                # reset
                mask = cat['TILEID']==tileid
                expid_list = np.array(cat['EXPID'][mask])
                subsets = []
                break
        if success:
            subsets.append(subset)
            mask = ~np.in1d(expid_list, np.concatenate(subsets))
            expid_list = expid_list[mask]
        if len(subsets)==n_sub:
            break
    
    print('tile', tileid, subsets)
    subsamp_dict[str(tileid)] = subsets
    for subset in subsets:
        mask = np.in1d(cat['EXPID'], subset)
        print(np.sum(mask), np.sum(cat['R_DEPTH_EBVAIR'][mask]))


80605  11   6132.75   1.53   1.80   1
tile 80605 [[73702, 74779, 68291, 74780, 68292, 74782, 67972]]
7 3946.492

80606  11   6628.39   1.66   1.95   1
tile 80606 [[67970, 68813, 68285, 68289, 68630, 67968, 67971]]
7 4045.1152

80607  17   9999.57   2.50   2.78   2
tile 80607 [[68847, 68845, 68028, 68664, 69440, 67767, 67768], [67744, 68662, 68666, 68846, 69441, 68844]]
7 4009.8706
6 4086.8523

80608  20  13990.97   3.50   3.89   3
tile 80608 [[68327, 68660, 69249, 67770, 68317], [68024, 68661, 68842, 68328, 67769, 68841], [69252, 68025, 69438, 68026, 67771, 69251]]
5 3909.7632
6 4053.0947
6 3959.572

80609  13   7011.71   1.75   2.06   2
tile 80609 [[68490, 68339, 68337, 68334, 68065, 68336], [67781, 68063, 68064, 67783, 68340, 68489, 68338]]
6 3523.5366
7 3488.1763

80610  15   9014.41   2.25   2.50   2
tile 80610 [[68332, 68478, 68041, 68330, 68488, 68683, 75114], [75113, 68477, 68040, 68333, 68331, 68042, 75116]]
7 3909.9048
7 3864.6833

80613   9   2384.14   3.97   4.42   4
tile 8

In [10]:
subsamp_dict

{'80605': [[73702, 74779, 68291, 74780, 68292, 74782, 67972]],
 '80606': [[67970, 68813, 68285, 68289, 68630, 67968, 67971]],
 '80607': [[68847, 68845, 68028, 68664, 69440, 67767, 67768],
  [67744, 68662, 68666, 68846, 69441, 68844]],
 '80608': [[68327, 68660, 69249, 67770, 68317],
  [68024, 68661, 68842, 68328, 67769, 68841],
  [69252, 68025, 69438, 68026, 67771, 69251]],
 '80609': [[68490, 68339, 68337, 68334, 68065, 68336],
  [67781, 68063, 68064, 67783, 68340, 68489, 68338]],
 '80610': [[68332, 68478, 68041, 68330, 68488, 68683, 75114],
  [75113, 68477, 68040, 68333, 68331, 68042, 75116]],
 '80613': [[69228, 68659],
  [68658, 69229],
  [68657],
  [69227, 69226, 69230, 69225]]}

In [11]:
# Print summary
for tileid_str in subsamp_dict.keys():
    tileid = int(tileid_str)
    mask = cat['TILEID']==tileid
    n_exp = np.sum(mask)
    tot_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    print('Tile {}:'.format(tileid))
    print('all:       n_exp={:2}  depth={:5.0f}s'.format(n_exp, tot_depth))
    subsets = subsamp_dict[tileid_str]
    for index, subset in enumerate(subsets):
        mask = np.in1d(cat['EXPID'], subset)
        subset_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
        print('subset {}:  n_exp={:2}  depth={:5.0f}s'.format(index+1, len(subset), subset_depth))
    mask = (cat['TILEID']==tileid) & (~np.in1d(cat['EXPID'], np.concatenate(subsets)))
    unused_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    print('unused:    n_exp={:2}  depth={:5.0f}s'.format(np.sum(mask), unused_depth))
    print()

Tile 80605:
all:       n_exp=11  depth= 6133s
subset 1:  n_exp= 7  depth= 3946s
unused:    n_exp= 4  depth= 2186s

Tile 80606:
all:       n_exp=11  depth= 6628s
subset 1:  n_exp= 7  depth= 4045s
unused:    n_exp= 4  depth= 2583s

Tile 80607:
all:       n_exp=17  depth=10000s
subset 1:  n_exp= 7  depth= 4010s
subset 2:  n_exp= 6  depth= 4087s
unused:    n_exp= 4  depth= 1903s

Tile 80608:
all:       n_exp=20  depth=13991s
subset 1:  n_exp= 5  depth= 3910s
subset 2:  n_exp= 6  depth= 4053s
subset 3:  n_exp= 6  depth= 3960s
unused:    n_exp= 3  depth= 2069s

Tile 80609:
all:       n_exp=13  depth= 7012s
subset 1:  n_exp= 6  depth= 3524s
subset 2:  n_exp= 7  depth= 3488s
unused:    n_exp= 0  depth=    0s

Tile 80610:
all:       n_exp=15  depth= 9014s
subset 1:  n_exp= 7  depth= 3910s
subset 2:  n_exp= 7  depth= 3865s
unused:    n_exp= 1  depth= 1240s

Tile 80613:
all:       n_exp= 9  depth= 2384s
subset 1:  n_exp= 2  depth=  581s
subset 2:  n_exp= 2  depth=  636s
subset 3:  n_exp= 1  depth

--------
## Require 2 subsamples for 80605 and 80606

In [12]:
np.random.seed(619)
nom_depth = 3000.

subsamp_dict_1 = {}

# tileid = 80607
for tileid in [80605, 80606]:
    
    print()
    
    mask = cat['TILEID']==tileid
    tot_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    # nom_depth = tot_depth / 2
    n_sub = 2
    print(tileid, '{:3} {:9.2f} {:3}'.format(np.sum(mask), tot_depth, n_sub))

    mask = cat['TILEID']==tileid
    expid_list = np.array(cat['EXPID'][mask])
    subsets = []

    total_iteration = 0
    while True:
        total_iteration += 1
        success = False
        iteration = 0
        while success==False:
            iteration += 1
            subset = []
            subset_depth = []
            while True:
                mask = ~np.in1d(expid_list, subset)
                expid = np.random.choice(expid_list[mask])
                subset.append(expid)
                subset_depth.append(cat['R_DEPTH_EBVAIR'][cat['EXPID']==expid][0])

                if total_iteration<200:
                    margin = 0.05
                elif ((total_iteration>200) & (total_iteration<400)):
                    margin = 0.1
                elif (total_iteration>400):
                    margin = 0.15
                if (np.sum(subset_depth)>(1-margin)*nom_depth) and (np.sum(subset_depth)<(1+margin)*nom_depth):
    #             if (np.sum(subset_depth)>0.8*nom_depth) and (np.sum(subset_depth)<1.2*nom_depth):
                    success = True
                    # print(np.sum(subset_depth))
                    break
                if len(subset)==len(expid_list):
                    break
            if iteration>(np.minimum(100, np.math.factorial(n_exp))):
                # reset
                mask = cat['TILEID']==tileid
                expid_list = np.array(cat['EXPID'][mask])
                subsets = []
                break
        if success:
            subsets.append(subset)
            mask = ~np.in1d(expid_list, np.concatenate(subsets))
            expid_list = expid_list[mask]
        if len(subsets)==n_sub:
            break
    
    print('tile', tileid, subsets)
    subsamp_dict_1[str(tileid)] = subsets
    for subset in subsets:
        mask = np.in1d(cat['EXPID'], subset)
        print(np.sum(mask), np.sum(cat['R_DEPTH_EBVAIR'][mask]))


80605  11   6132.75   2
tile 80605 [[74779, 73702, 68290, 74781, 74783, 67975], [67972, 74782, 68292, 74780, 68291]]
6 3023.1863
5 3109.5645

80606  11   6628.39   2
tile 80606 [[68285, 68284, 67971, 68630], [68288, 68812, 68813, 67970, 68289]]
4 3077.1677
5 3001.5876


In [13]:
# subsamp_dict['80605'] = subsamp_dict_1['80605']
# subsamp_dict['80606'] = subsamp_dict_1['80606']

In [14]:
subsamp_dict_1

{'80605': [[74779, 73702, 68290, 74781, 74783, 67975],
  [67972, 74782, 68292, 74780, 68291]],
 '80606': [[68285, 68284, 67971, 68630], [68288, 68812, 68813, 67970, 68289]]}

In [15]:
# Print summary
for tileid_str in subsamp_dict_1.keys():
    tileid = int(tileid_str)
    mask = cat['TILEID']==tileid
    n_exp = np.sum(mask)
    tot_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    print('Tile {}:'.format(tileid))
    print('all:       n_exp={:2}  depth={:5.0f}s'.format(n_exp, tot_depth))
    subsets = subsamp_dict_1[tileid_str]
    for index, subset in enumerate(subsets):
        mask = np.in1d(cat['EXPID'], subset)
        subset_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
        print('subset {}:  n_exp={:2}  depth={:5.0f}s'.format(index+1, len(subset), subset_depth))
    mask = (cat['TILEID']==tileid) & (~np.in1d(cat['EXPID'], np.concatenate(subsets)))
    unused_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    print('unused:    n_exp={:2}  depth={:5.0f}s'.format(np.sum(mask), unused_depth))
    print()

Tile 80605:
all:       n_exp=11  depth= 6133s
subset 1:  n_exp= 6  depth= 3023s
subset 2:  n_exp= 5  depth= 3110s
unused:    n_exp= 0  depth=    0s

Tile 80606:
all:       n_exp=11  depth= 6628s
subset 1:  n_exp= 4  depth= 3077s
subset 2:  n_exp= 5  depth= 3002s
unused:    n_exp= 2  depth=  550s

